# 10 — Train VAE

Short training run using `poregen` library modules.  
Run on an interactive HPC node with GPU(s).

In [ ]:
from poregen.configs.config import load_config
from pathlib import Path
from torch.utils.data import DataLoader
import torch

from poregen.dataset.loader import PatchDataset
from poregen.models.vae import build_vae
from poregen.losses import compute_total_loss
from poregen.training import (
    seed_everything,
    select_device,
    get_autocast_dtype,
    make_scaler,
    train_loop,
)

## Config

In [ ]:
cfg = load_config("../src/poregen/configs/vae_default.yaml")

# DGX Spark overrides
cfg["data"]["num_workers"] = 12
cfg["data"]["batch_size"]  = 16   # increase if VRAM allows

print(cfg)

## Setup

In [ ]:
seed_everything(cfg["training"]["seed"])
device = select_device()  # or select_device(gpu_id=1)
autocast_dtype = get_autocast_dtype(device)
scaler = make_scaler(device)
print(f"Device: {device}  |  AMP dtype: {autocast_dtype}")

## Data

In [ ]:
DATA_ROOT = Path("../data/processed")
INDEX     = DATA_ROOT / "patch_index.parquet"
STATS     = DATA_ROOT / "volume_stats.json"

train_ds = PatchDataset(INDEX, DATA_ROOT, split="train", stats_path=STATS)
val_ds   = PatchDataset(INDEX, DATA_ROOT, split="val",   stats_path=STATS)
test_ds  = PatchDataset(INDEX, DATA_ROOT, split="test",  stats_path=STATS)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg["data"]["batch_size"],
    shuffle=True,
    num_workers=cfg["data"]["num_workers"],
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg["data"]["batch_size"],
    shuffle=False,
    num_workers=cfg["data"]["num_workers"],
    pin_memory=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg["data"]["batch_size"],
    shuffle=False,
    num_workers=cfg["data"]["num_workers"],
    pin_memory=True,
)

print(f"Train patches: {len(train_ds):,}  |  Val patches: {len(val_ds):,}  |  Test patches: {len(test_ds):,}")

## Model + Optimizer

In [ ]:
model = build_vae(
    cfg["model"]["name"],
    z_channels=cfg["model"]["z_channels"],
    base_channels=cfg["model"]["base_channels"],
    n_blocks=cfg["model"]["n_blocks"],
    patch_size=cfg["model"]["patch_size"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg["training"]["lr"],
    weight_decay=cfg["training"]["weight_decay"],
)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

_t = cfg["training"]
if _t.get("scheduler", "none") == "cosine":
    warmup = LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=_t["warmup_steps"],
    )
    cosine = CosineAnnealingLR(
        optimizer,
        T_max=_t["total_steps"] - _t["warmup_steps"],
        eta_min=_t["lr_min"],
    )
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[_t["warmup_steps"]],
    )
    print(f"Scheduler: linear warmup {_t['warmup_steps']} steps → cosine decay to {_t['lr_min']:.0e}")
else:
    scheduler = None
    print("Scheduler: none")

## Loss function

In [ ]:
loss_fn = lambda output, batch, step: compute_total_loss(output, batch, step, cfg)

from torch.utils.tensorboard import SummaryWriter

EXP_NAME  = "conv_baseline"  # change per experiment
tb_writer = SummaryWriter(f"../runs/vae/{EXP_NAME}/tb")

history = train_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    scaler,
    loss_fn,
    total_steps=cfg["training"]["total_steps"],
    log_every=cfg["training"]["log_every"],
    eval_every=cfg["training"]["eval_every"],
    val_batches=cfg["training"]["val_batches"],
    test_loader=test_loader,
    test_every=cfg["training"]["test_every"],
    test_batches=cfg["training"]["test_batches"],
    save_every=cfg["training"]["save_every"],
    image_log_every=cfg["training"]["image_log_every"],
    sample_every=cfg["training"]["sample_every"],
    n_patch_samples=cfg["training"]["n_patch_samples"],
    run_dir=f"../runs/vae/{EXP_NAME}",
    device=device,
    autocast_dtype=autocast_dtype,
    max_grad_norm=cfg["training"]["max_grad_norm"],
    scheduler=scheduler,
    tb_writer=tb_writer,
)

tb_writer.close()
print(f"Done. Final loss: {history[-1]['total']:.4f}")

In [ ]:
from torch.utils.tensorboard import SummaryWriter

EXP_NAME  = "conv_baseline"  # change per experiment
tb_writer = SummaryWriter(f"../runs/vae/{EXP_NAME}/tb")

history = train_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    scaler,
    loss_fn,
    total_steps=cfg["training"]["total_steps"],
    log_every=cfg["training"]["log_every"],
    eval_every=cfg["training"]["eval_every"],
    val_batches=cfg["training"]["val_batches"],
    test_loader=test_loader,
    test_every=cfg["training"]["test_every"],
    test_batches=cfg["training"]["test_batches"],
    save_every=cfg["training"]["save_every"],
    image_log_every=cfg["training"]["image_log_every"],
    run_dir=f"../runs/vae/{EXP_NAME}",
    device=device,
    autocast_dtype=autocast_dtype,
    max_grad_norm=cfg["training"]["max_grad_norm"],
    tb_writer=tb_writer,
)

tb_writer.close()
print(f"Done. Final loss: {history[-1]['total']:.4f}")

## Quick loss plot

In [ ]:
import matplotlib.pyplot as plt

train_h = [r for r in history if r["split"] == "train"]
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot([r["step"] for r in train_h], [r["total"] for r in train_h], lw=0.6)
ax.set(xlabel="Step", ylabel="Total loss", title="Training loss")
plt.tight_layout()
plt.show()